In [1]:
import getpass 
import os
from langchain_cohere import ChatCohere
from langchain_cohere import ChatCohere
from langchain_core.prompts import ChatPromptTemplate, PromptTemplate, MessagesPlaceholder 
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage
from langchain_core.runnables import RunnableLambda, RunnablePassthrough, RunnableBranch, RunnableSequence, RunnableParallel 
from langchain_core.output_parsers import StrOutputParser

ModuleNotFoundError: No module named 'langchain_cohere'

In [ ]:
if not os.getenv("COHERE_API_KEY"):
    os.environ["COHERE_API_KEY"] = getpass.getpass("Enter your Cohere API key: ")

In [ ]:
llm = ChatCohere(model = "command-r7b-12-2024", temperature = 0.6, max_tokens = 400) 
llm.invoke('hi')

In [ ]:
message = [SystemMessage("You are a financial expert. Recommend investment stratergies to the user"), 
           HumanMessage("Tell me best investment stratergies in 2-3 lines")]
message

In [ ]:
llm.invoke(message)

## Prompt templates

In [ ]:
chat = PromptTemplate.from_template("Let's have a debate on the topic: {topic}") 
res = chat.invoke({'topic':'Geo-politics'}) 
llm.invoke(res).content

### ChatPromptTemplate

In [ ]:
chat = ChatPromptTemplate([ 
    ("system", "You are friendly AI assistat. Answer in 2-3 lines on what the user asks"), 
    ("user","Define the following finance term: {finance_term}")]) 
res = chat.invoke({'finance_term':'repo_rate'}) 
llm.invoke(res).content

In [ ]:
chat = ChatPromptTemplate([ 
    #("system", "You are caveman who answers in only keywords. Answer the user query based on how caveman responds{some_input}"), 
    ("user","What is a fun fact about {place}")])  
res = chat.invoke({'place':'Ladakh'}) 
llm.invoke(res).content

### Messages Placeholder

In [ ]:
chat_1 = ChatPromptTemplate.from_messages([ 
    ('system', "You are friendly assistant bot. Answer the queries in 2-3 lines"), 
    MessagesPlaceholder(variable_name = 'history'), 
    ("user", "{input}")])

In [ ]:
chat_history = [
    HumanMessage(content = "My name is Nick"), 
    AIMessage(content = 'Nice to meet you Nick')
]

In [ ]:
res = chat_1.invoke({ 
    "history":chat_history, 
    "input":"What is my name?"})

In [ ]:
llm.invoke(res).content

In [ ]:
#verify whether the messages are stored or not 
chain = chat_1 | llm 
chat_history = [] 
def chat_with_verify(user_input): 
    global chat_history 
    res = chain.invoke({ 
    'history': chat_history, 
    'input':user_input}) 
    chat_history.append(HumanMessage(content = user_input)) 
    chat_history.append(AIMessage(content = res.content)) 
    print(f"\n > User: {user_input}") 
    print(f"> AI: {res.content}") 
    print(f"\n [Verification] Current history length: {len(chat_history)} messages") 
    for i, msg in enumerate(chat_history): 
        print(f" Message {i}: {msg.type} --> {msg.content[:20]}...")



In [ ]:
chat_with_verify("Hi, I'm Bob")

In [ ]:
chat_with_verify("Wait, what was my name")

### Runnables and Chains 

In [ ]:
import getpass 
import os
from langchain_cohere import ChatCohere
from langchain_cohere import ChatCohere
from langchain_core.prompts import ChatPromptTemplate, PromptTemplate, MessagesPlaceholder 
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage
from langchain_core.runnables import RunnableLambda, RunnablePassthrough, RunnableBranch, RunnableSequence, RunnableParallel 
from langchain_core.output_parsers import StrOutputParser

In [ ]:
llm.invoke('hi')

In [ ]:
#Runnable sequence

In [ ]:
joke_prompt = PromptTemplate.from_template("Write a joke about {topic}") 
explain_prompt = PromptTemplate.from_template("Explain the following joke: {text}")
parser = StrOutputParser()

In [ ]:
joke_generator = joke_prompt | llm | parser
joke_explainer = explain_prompt | llm | parser

In [ ]:
full_chain = joke_generator | joke_explainer | parser 
full_chain.invoke({'topic':'git'})

In [ ]:
full_chain_2 = RunnableSequence(joke_generator, {'text':RunnablePassthrough()}, joke_explainer) 
full_chain_2.invoke({'topic':'git'})

In [ ]:
##Runnable Parallel 
prompt_1 = PromptTemplate.from_template("Generate a blog about {topic_1}") 
prompt_2 = PromptTemplate.from_template("Generate a insta post about {topic_2}") 

In [ ]:
parallel_chain = RunnableParallel({ 
    'blog':prompt_1 | llm | parser, 
    "insta_post": prompt_2 | llm | parser})

In [ ]:
parallel_chain.invoke({'topic_1':'git','topic_2':'Travelling'})

In [ ]:
#RunnableLambda 
prompt_temp = ChatPromptTemplate.from_messages([ 
    ("system", "You are professional editor, your goal is to be concise and accurate"), 
    ("user", "The user provided this text: {cleaned_text}. It is {count} words long. Summarize it")])

In [ ]:
def analyze_text(text:str): 
    cleaned = text.strip() 
    word_count = len(cleaned.split()) 
    return {'cleaned_text': cleaned, 'count': word_count}

In [ ]:
text_processor = RunnableLambda(analyze_text) 
chain = text_processor | prompt_temp | llm | parser 
chain.invoke("            runnables are powerful tools for developers                    ")

In [ ]:
#Runnable Branch 
math_chain = PromptTemplate.from_template("You are an expert mathematician. Solve this problem step by step :{input}") | llm | parser 
phy_chain = PromptTemplate.from_template("You are an expert physicist. Explain the concept clearly in 4-5 lines: {input}") | llm | parser 
gen_chain = PromptTemplate.from_template("You are an helpful assistant. Answer the query: {input}") | llm | parser

In [ ]:
classifier_chain = ChatPromptTemplate.from_messages([ 
    ("system", "Classify the user's question into either as'math', 'physics', or 'general'. Respond only with one word: the topic name"), 
    ("user", '{input}')]) | llm | parser

In [ ]:
def is_math_topic(input_data): 
    return "math" in input_data['topic'].lower() 

def is_phy_topic(input_data): 
    return "physics" in input_data['topic'].lower()

In [ ]:
branch = RunnableBranch(
    (is_math_topic, math_chain), 
    (is_phy_topic, phy_chain), 
    gen_chain)

In [ ]:
full_chain = {'topic': classifier_chain, 'input':RunnablePassthrough()} | branch

In [ ]:
full_chain.invoke({'input': 'What is the square of 144 plus 5?'})

In [ ]:
full_chain.invoke({'input': "Explain how time dilation works in black hole"})

In [ ]:
full_chain.invoke({'input':'What is the capital of France ?'})